# 🎮 Jogo PCP 2 — FLAMENGO — Cockpit

Fluxo: 1) preencha **inputs**; 2) **preview** do que o prof mandou; 3) **rodar pipeline**; 4) **cockpit de factibilidade**; 5) **histórico**.

## 1. Setup (rodar sempre)

In [1]:
import sys, os, json, copy
from pathlib import Path
BASE = Path('..').resolve()
sys.path.insert(0, str(BASE))
os.chdir(BASE)
from src.pipeline import run_rodada
from src.cockpit_view import imprimir_cockpit, preview_rodada_xlsm
from src.dashboard import plot_historico, tabela_resumo, ler_snapshots
print('OK — base:', BASE)

OK — base: C:\Users\lolil\Downloads\Jogo PCP 2 (a vinganca)


## 2. Configurar rodada

Troque `RODADA` a cada nova rodada. O arquivo `Rodada N.xlsm` deve estar em `rodadas/` (entregue pelo professor).

In [ ]:
RODADA = 2
RODADA_PATH = BASE / 'rodadas' / f'Rodada {RODADA}.xlsm'
assert RODADA_PATH.exists(), f'Arquivo nao encontrado: {RODADA_PATH}'

## 3. Preview do que o professor mandou

In [ ]:
preview_rodada_xlsm(RODADA_PATH)

## 4. Preencher OPs e Preços da rodada

**OPS**: lista de pedidos do varejo. Cada dict = (cidade, pa, qtd, dia_entrega).

**PRECOS**: preço de mercado por PA na rodada (o professor passa).

In [ ]:
OPS = [
    {'cidade': 'São Paulo',      'pa': 'PA1', 'qtd': 50000, 'dia_entrega': 5},
    {'cidade': 'Rio de Janeiro', 'pa': 'PA2', 'qtd': 30000, 'dia_entrega': 4},
    {'cidade': 'Belo Horizonte', 'pa': 'PA3', 'qtd': 20000, 'dia_entrega': 5},
    # adicione mais linhas...
]

PRECOS = {'PA1': 78.50, 'PA2': 51.20, 'PA3': 24.80}

# margem mínima pra atender (em %). 0 = só recusa margem negativa. 
# Coloque 10 se quiser recusar OPs com margem < 10%.
MARGEM_MIN_PCT = 0.0

print(f'OPs: {len(OPS)} | Precos: {PRECOS} | Margem min: {MARGEM_MIN_PCT}%')

## 5. Rodar pipeline (planner + LP + factibilidade)

In [ ]:
resumo = run_rodada(
    rodada_n=RODADA,
    rodada_xlsm_path=RODADA_PATH,
    ops=OPS,
    precos=PRECOS,
    margem_minima_pct=MARGEM_MIN_PCT,
)

## 6. Cockpit de factibilidade

In [ ]:
imprimir_cockpit(resumo['cockpit'])

## 7. O que foi escrito em FLAMENGO.xlsm (para entregar)

In [ ]:
import pandas as pd
from src.io_xlsm import ler_sol_transp
items = ler_sol_transp(BASE / 'rodadas' / 'FLAMENGO.xlsm', rodada=RODADA)
if items:
    df = pd.DataFrame([{
        'Origem': t.origem_tipo, 'Cidade Or': t.origem_cidade,
        'Dia': t.dia_part, 'Modal': t.modal, 'Item': t.item, 'Qtd': t.qtd,
        'Cidade Dest': t.destino_cidade, 'Chega': f'R{t.rod_cheg}D{t.dia_cheg}',
    } for t in items])
    print(df.to_string(index=False))
else:
    print('Nada escrito na rodada')

## 8. Histórico (rodadas anteriores)

In [ ]:
plot_historico(BASE / 'estado')